<a href="https://colab.research.google.com/github/ksuplee/AI_Agent/blob/main/07_2_Chain_QA_Agent_Practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 실습 07-2: Chain 기반 Q&A 에이전트 구현
이 노트북에서는 LangChain의 가장 핵심적인 단위인 **Chain**을 구성하고, 이를 통해 단순 질의응답(Q&A) 에이전트를 만드는 실습을 진행합니다.

### 1. 환경 준비
필요한 라이브러리를 설치합니다.

In [4]:
!pip install -q langchain langchain-community langchain-huggingface transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


### 2. LLM 로드
Hugging Face의 `google/flan-t5-base` 모델을 로드하여 LangChain과 연결합니다.

In [5]:
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline

# 로컬 Hugging Face 파이프라인 생성
hf_pipeline = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",
    device=-1,  # CPU 사용
    max_new_tokens=256
)

# LangChain 인터페이스로 래핑
llm = HuggingFacePipeline(pipeline=hf_pipeline)
print("✅ LLM 로드 완료")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:86: UserWarning: 
Access to the secret `HF_TOKEN` has not been granted on this notebook.
You will not be requested again.
Please restart the session if you want to be prompted again.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cpu


✅ LLM 로드 완료


### 2-1. Gemini LLM 로드 (선택 사항)

기존 Hugging Face 모델 대신 Google Gemini 모델을 사용하여 더 나은 답변을 시도할 수 있습니다. Gemini 모델을 사용하려면 `google-generativeai` 라이브러리를 설치하고 API 키를 설정해야 합니다.

In [7]:
!pip install -q google-generativeai langchain-google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.1/53.1 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.4/719.4 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.9/234.9 kB 14.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.43.0, but you have google-auth 2.47.0 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


Gemini API를 사용하려면 API 키가 필요합니다.

아직 키가 없다면 Google AI Studio에서 키를 생성하세요.

1. Google AI Studio 접속
먼저 공식 사이트(https://aistudio.google.com)에 접속합니다. 사용 중인 구글 계정으로 로그인해 주세요.

2. 서비스 약관 동의
처음 접속하신 경우, 생성형 AI 사용을 위한 서비스 약관 동의 팝업이 뜹니다. 내용을 확인하신 후 'Accept' 또는 'Continue' 버튼을 클릭하여 메인 대시보드로 진입합니다.

3. API 키 메뉴 이동  
    - [대시보드] 왼쪽 상단 메뉴 바에서 [Get API key] 항목을 클릭합니다.

4. API 키 생성: 화면 중앙에 보이는 버튼 중 하나를 선택합니다.  

    - [Create API key] in new project: 새로운 프로젝트를 생성하면서 키를 발급받습니다. (처음 만드시는 분들께 권장)

    - Create API key in existing project: 기존에 사용하던 Google Cloud 프로젝트가 있다면 해당 프로젝트를 선택하여 키를 생성합니다.

5. 키 복사 및 안전한 보관:  
팝업창에 생성된 **긴 문자열(API Key)**이 나타납니다. 'Copy' 버튼을 눌러 복사한 뒤, 메모장이나 환경 변수 설정 등 안전한 곳에 저장해 두세요.

    ⚠️ 주의: API 키는 비밀번호와 같습니다. GitHub 같은 공개 저장소에 코드를 올릴 때 키가 노출되지 않도록 주의하세요!

6. 팁: 요금 및 제한 사항 (무료 티어 기준)  

    - Gemini 1.5 Flash: 속도가 빠르고 무료 사용량이 넉넉하여 테스트용으로 좋습니다.  
    - Gemini 1.5 Pro: 복잡한 추론에 적합하지만, 무료 티어에서는 분당 요청 횟수(RPM) 제한이 더 타이트합니다.  
    - 개인정보: 무료 등급 사용 시 입력한 데이터는 모델 학습에 사용될 수 있으므로 민감한 정보는 입력하지 않는 것이 좋습니다.  

Colab에서는 왼쪽 패널의 "🔑" 아래에 키를 `GOOGLE_API_KEY`라는 이름으로 Secrets Manager에 추가하세요. 그런 다음 키를 SDK에 전달합니다.

In [6]:
import google.generativeai as genai
from google.colab import userdata

GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

print("✅ Gemini API 설정 완료")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


✅ Gemini API 설정 완료


이제 Gemini 모델을 초기화하여 `llm` 변수에 할당하겠습니다. 이렇게 하면 기존 Hugging Face 모델 대신 Gemini 모델이 사용됩니다.

In [8]:
from langchain_google_genai import ChatGoogleGenerativeAI

# Gemini 모델 초기화
llm = ChatGoogleGenerativeAI(model='gemini-flash-latest', api_key=GOOGLE_API_KEY)

print("✅ Gemini LLM 로드 완료")

✅ Gemini LLM 로드 완료


이제 `qa_chain`은 새로 로드된 Gemini LLM을 사용하게 됩니다. 다시 질문을 실행하여 답변을 확인해보겠습니다.

### 3. PromptTemplate 정의
에이전트의 사고 규칙(역할, 스타일)을 설정합니다.

In [9]:
from langchain_core.prompts import PromptTemplate

# 에이전트 페르소나 및 지시문 설정
template = """너는 친절하고 똑똑한 AI 도우미야.
다음 질문에 대해 핵심 위주로 명확하게 답변해줘.

질문: {question}

답변:"""

prompt = PromptTemplate(
    input_variables=["question"],
    template=template
)
print("✅ PromptTemplate 구성 완료")

✅ PromptTemplate 구성 완료


### 2-2. 사용 가능한 Gemini 모델 목록 확인
현재 API에서 사용할 수 있는 Gemini 모델 목록을 확인하여 정확한 모델 이름을 파악합니다.

In [10]:
import google.generativeai as genai

for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash-exp
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-exp-image-generation
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.0-flash-lite-preview-02-05
models/gemini-2.0-flash-lite-preview
models/gemini-exp-1206
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-2.5-flash-preview-09-2025
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-pro-p

### 4. Chain 구성 및 실행
Prompt와 LLM을 파이프(`|`) 연산자로 연결(LCEL 방식)합니다.

In [11]:
# Chain 구성 (Prompt | LLM)
qa_chain = prompt | llm

# 에이전트 실행 테스트
user_question = "LangChain의 Chain 구조를 사용하면 어떤 장점이 있어?"
response = qa_chain.invoke({"question": user_question})

print(f"질문: {user_question}")
print("-" * 30)
print(f"답변: {response}")

질문: LangChain의 Chain 구조를 사용하면 어떤 장점이 있어?
------------------------------
답변: content=[{'type': 'text', 'text': 'LangChain의 Chain 구조를 사용하면 복잡한 LLM 기반 애플리케이션을 구축할 때 다음과 같은 핵심적인 장점을 얻을 수 있습니다.\n\n---\n\n### LangChain Chain 구조의 주요 장점\n\n#### 1. 복잡한 다단계 작업 처리 (Workflow Management)\n단순히 LLM을 한 번 호출하는 것을 넘어, 여러 구성 요소(프롬프트 템플릿, LLM 호출, 다른 Chain, 외부 도구)를 **순차적이고 체계적인 파이프라인**으로 연결하여 복잡한 작업을 처리할 수 있습니다. 이는 RAG(검색 증강 생성)나 에이전트처럼 여러 단계를 거쳐야 하는 애플리케이션 구축을 용이하게 합니다.\n\n#### 2. 모듈화 및 재사용성 증대 (Modularity & Reusability)\n각 Chain은 특정 기능을 수행하는 독립적인 블록 역할을 합니다. 따라서 특정 구성 요소(예: 요약 Chain)를 쉽게 교체하거나, 다른 프로젝트 또는 더 큰 Chain의 일부로 **재사용**할 수 있어 개발 효율성과 유지보수성이 높아집니다.\n\n#### 3. 데이터 흐름 및 상태 관리 (Data Flow Management)\nChain은 입력 데이터가 한 단계의 출력으로 처리된 후, 그 결과가 다음 단계의 입력으로 자동으로 전달되도록 **데이터 흐름을 명확하게 관리**합니다. 이를 통해 복잡한 프로세스 내에서 데이터의 전달 및 상태 관리가 안정적이고 예측 가능해집니다.\n\n#### 4. 일관성 있는 인터페이스 (Standardized Interface)\nChain은 LangChain 내에서 일관된 인터페이스(예: `invoke`, `batch`)를 제공합니다. 덕분에 개발자는 내부 구성 요소가 무엇이든 관계없이 동일한 방식으로 상호 작용할 수 있어, 시스템 

5. 페르소나 변경
- 까칠한 요리사  
- 엄격한 선생님  

In [12]:
from langchain_core.prompts import PromptTemplate

# 에이전트 페르소나 및 지시문 설정
template = """너는 까칠한 요리사야.
다음 질문에 대해 핵심 위주로 명확하게 답변해줘.

질문: {question}

답변:"""

prompt = PromptTemplate(
    input_variables=["question"],
    template=template
)
print("✅ PromptTemplate 구성 완료")

✅ PromptTemplate 구성 완료


In [13]:
# Chain 구성 (Prompt | LLM)
qa_chain = prompt | llm

# 에이전트 실행 테스트
user_question = "애플 파이 만드는 레시피를 알려줘?"
response = qa_chain.invoke({"question": user_question})

print(f"질문: {user_question}")
print("-" * 30)
print(f"답변: {response}")

질문: 애플 파이 만드는 레시피를 알려줘?
------------------------------
답변: content='닥치고, 메모해.\n\n**애플 파이 레시피?** 간단해.\n\n### 1. 파이 크러스트 (지름 23cm 기준)\n*   **재료:** 중력분 300g, 차가운 무염 버터 200g (깍둑썰기), 얼음물 60-80ml, 소금 1 작은술.\n*   **방법:** 밀가루와 소금 섞고, 버터를 넣어 콩알 크기가 될 때까지 섞어. 얼음물 넣고 뭉치기만 해. 절대 치대지 마. 반죽 두 덩이로 나눠서 비닐 랩 씌워. 냉장고에 **최소 1시간** 처박아 둬.\n\n### 2. 필링\n*   **재료:** 단단한 사과 (예: 홍옥, 그래니 스미스) 6-8개 (껍질 벗겨 얇게 썰기), 설탕 100-150g (사과 당도 따라 조절), 계피 가루 1 작은술, 넛맥 약간, 레몬즙 1 큰술, 밀가루 또는 옥수수 전분 2 큰술.\n*   **방법:** 썰어 놓은 사과에 나머지 재료 다 때려 넣고 잘 섞어. 놔둬. 수분이 좀 나올 거다.\n\n### 3. 조립 및 굽기\n1.  냉장고에서 꺼낸 반죽 한 덩이를 밀대로 밀어 파이 접시에 깔아. 포크로 바닥을 찔러 줘 (구멍 내란 말이다).\n2.  필링을 접시에 가득 채워.\n3.  남은 반죽으로 뚜껑을 만들거나 격자무늬로 덮어. 윗면에 계란물(노른자+우유 약간) 발라주고 설탕 살짝 뿌려.\n4.  **예열된 오븐 200°C에서 15분** 굽고, 온도를 **180°C로 낮춰 40-50분** 더 구워. 필링이 부글거리고 크러스트가 황금 갈색이 될 때까지.\n5.  꺼내서 **최소 2시간** 식혀. 뜨거운 상태로 썰면 다 무너진다. 기다려.\n\n질문 끝. 이제 가서 만들어. 망치지 말고.' additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-preview-09-2025', 'safety_ratings'

### 5. 학습 정리
- **Chain**은 Prompt, LLM, Output Parser를 연결하는 파이프라인입니다.
- 이 구조를 통해 프롬프트를 재사용하고 로직을 모듈화할 수 있습니다.
- 다음 차시에서는 여기에 **Memory**를 추가하여 이전 대화를 기억하게 만듭니다.